In [7]:
import numpy as np
import mlflow
import azureml.core

# display the core SDK version number
print("Azure ML SDK Version: ", azureml.core.VERSION)

Azure ML SDK Version:  1.61.0


In [8]:
from azureml.core import Workspace
from azureml.core.model import Model

subscription_id = 'f2f70602-65d9-479b-8014-a1c92a06ef0e'
resource_group = 'Learn_MLOps'
workspace_name = 'MLOps'

ws = Workspace(subscription_id, resource_group, workspace_name)
print(ws.name, ws.resource_group, ws.location, sep='\n')

MLOps
Learn_MLOps
eastus2


In [9]:
mlflow.set_tracking_uri(ws.get_mlflow_tracking_uri())

## Deploy to ACI

In [10]:
from azureml.core.webservice import AciWebservice, Webservice

# Set the model path to the model folder created by your run
model_path = "azureml://experiments/mlflow-support-vector-machine/runs/281b2562-0616-4251-9384-92a8b6d75661/artifacts"

In [11]:
# Configure 
aci_config = AciWebservice.deploy_configuration(cpu_cores=1, 
                                                memory_gb=1, 
                                                tags={'method' : 'sklearn'}, 
                                                description='weather pred model',
                                                location='northeurope')

#### register and deploy the model in one step with the Azure Machine Learning SDK deploy method.

In [18]:
import os
from azureml.core.webservice import AciWebservice, Webservice
from azureml.core.model import Model, InferenceConfig
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies

# ── Step 1: load both registered models (mirrors AKS deployment) ────────────

model1 = Model(ws, name="scaler")
model2 = Model(ws, name="support-vector-classifier")
print(f"scaler v{model1.version}, support-vector-classifier v{model2.version}")

# ── Step 2: scoring script — ONNX + joblib scaler ───────────────────────────

score_script = '''\
import numpy as np, os, glob, joblib, onnxruntime
from inference_schema.schema_decorators import input_schema, output_schema
from inference_schema.parameter_types.numpy_parameter_type import NumpyParameterType

def init():
    global model, scaler, input_name, label_name

    scaler_path = glob.glob(
        os.path.join(os.getenv("AZUREML_MODEL_DIR"), "scaler", "**", "*.pkl"),
        recursive=True
    )[0]
    scaler = joblib.load(scaler_path)

    model_onnx = glob.glob(
        os.path.join(os.getenv("AZUREML_MODEL_DIR"), "support-vector-classifier", "**", "*.onnx"),
        recursive=True
    )[0]
    model = onnxruntime.InferenceSession(model_onnx, None)
    input_name  = model.get_inputs()[0].name
    label_name  = model.get_outputs()[0].name

@input_schema("data", NumpyParameterType(np.array([[34.927778, 0.24, 7.3899, 83, 16.1000, 1016.51, 1]])))
@output_schema(NumpyParameterType(np.array([0])))
def run(data):
    try:
        data   = scaler.transform(data.reshape(1, 7))
        result = model.run([label_name], {input_name: data.astype(np.float32)})[0]
    except Exception as e:
        return str(e)
    return result.tolist()
'''

with open("score.py", "w") as f:
    f.write(score_script)
print("score.py written.")

# ── Step 3: environment ──────────────────────────────────────────────────────

env = Environment(name="weather-deploy-env")
conda_deps = CondaDependencies()
conda_deps.set_python_version("3.8")
for pkg in ["numpy", "onnxruntime", "joblib", "scikit-learn",
            "inference-schema[numpy-support]", "azureml-defaults"]:
    conda_deps.add_pip_package(pkg)
env.python.conda_dependencies = conda_deps

# ── Step 4: inference config ─────────────────────────────────────────────────

inference_config = InferenceConfig(entry_script="score.py", environment=env)

# ── Step 5: deploy to ACI ────────────────────────────────────────────────────

webservice = Model.deploy(
    workspace=ws,
    name="port-weather-pred",
    models=[model1, model2],
    inference_config=inference_config,
    deployment_config=aci_config,
    overwrite=True
)
webservice.wait_for_deployment(show_output=True)

model = model2
print("Scoring URI:", webservice.scoring_uri)


scaler v1, support-vector-classifier v3
score.py written.


/tmp/ipykernel_89396/967336229.py:68: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  webservice = Model.deploy(


Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2026-05-06 20:17:36-07:00 Creating Container Registry if not exists.
2026-05-06 20:17:37-07:00 Use the existing image.
2026-05-06 20:17:37-07:00 Generating deployment configuration.
2026-05-06 20:17:42-07:00 Submitting deployment to compute.
2026-05-06 20:17:45-07:00 Checking the status of deployment port-weather-pred..
2026-05-06 20:30:46-07:00 Checking the status of inference endpoint port-weather-pred.
Succeeded
ACI service creation operation finished, operation "Succeeded"
Scoring URI: http://313197b4-0409-4df0-81a3-09a164b89bf2.northeurope.azurecontainer.io/score


In [19]:
# ── Test the deployed service ────────────────────────────────────────────────
import json
from azureml.core.webservice import AciWebservice

webservice = AciWebservice(ws, "port-weather-pred")

input_payload = json.dumps({
    "data": [[34.927778, 0.24, 7.3899, 83, 16.1000, 1016.51, 1]]
})

output = webservice.run(input_payload)
print("Prediction:", output)


Prediction: [1]
